In [ ]:
# Import des bibliothèques nécessaires
import numpy as np
from tensorflow.keras.models import Sequential  # Pour créer un modèle séquentiel
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D  # Couches du réseau
from tensorflow.keras import backend as K  # Pour gérer le backend de Keras
from tensorflow.keras.datasets import mnist  # Jeu de données MNIST
from tensorflow.keras.utils import to_categorical  # Pour convertir les labels en one-hot encoding

# Définir le format des dimensions des images : 'channels_first' (1, 28, 28) ou 'channels_last' (28, 28, 1)
K.set_image_data_format('channels_first')

# Fixer une graine aléatoire pour la reproductibilité des résultats
seed = 7
np.random.seed(seed)

# Charger les données MNIST (images de chiffres manuscrits 28x28)
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Redimensionner les données pour les adapter au format 'channels_first' (batch, 1, 28, 28)
# et normaliser les valeurs des pixels entre 0 et 1
X_train = X_train.reshape(X_train.shape[0], 1, 28, 28).astype('float32') / 255
X_test = X_test.reshape(X_test.shape[0], 1, 28, 28).astype('float32') / 255

# Convertir les labels (0-9) en one-hot encoding (ex. : 3 → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0])
y_train = to_categorical(y_train, num_classes=10)
y_test = to_categorical(y_test, num_classes=10)
num_classes = 10  # Nombre de classes (chiffres de 0 à 9)

# Définition du modèle CNN
def large_model():
    # Créer un modèle séquentiel (empilement linéaire de couches)
    model = Sequential()

    # Ajouter une couche de convolution 2D avec 30 filtres de taille 5x5,
    # fonction d'activation ReLU, et input_shape pour la première couche
    model.add(Conv2D(30, (5, 5), input_shape=(1, 28, 28), activation='relu'))

    # Ajouter une couche de max-pooling pour réduire la taille spatiale (divise par 2)
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Ajouter une deuxième couche de convolution avec 15 filtres de taille 3x3
    model.add(Conv2D(15, (3, 3), activation='relu'))

    # Ajouter une deuxième couche de max-pooling
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Ajouter une couche de dropout pour réduire le surapprentissage (20% des neurones désactivés aléatoirement)
    model.add(Dropout(0.2))

    # Aplatir les données pour les passer aux couches denses (fully connected)
    model.add(Flatten())

    # Ajouter une couche dense (128 neurones) avec activation ReLU
    model.add(Dense(128, activation='relu'))

    # Ajouter une deuxième couche dense (50 neurones) avec activation ReLU
    model.add(Dense(50, activation='relu'))

    # Ajouter la couche de sortie avec 10 neurones (un par classe) et activation softmax
    model.add(Dense(num_classes, activation='softmax'))

    # Compiler le modèle : définir la fonction de perte, l'optimiseur et les métriques
    model.compile(loss='categorical_crossentropy',  # Fonction de perte pour la classification multi-classe
                  optimizer='adam',  # Optimiseur (adaptatif et efficace)
                  metrics=['accuracy'])  # Métrique pour évaluer la performance

    return model  # Retourner le modèle construit

# Construire le modèle en appelant la fonction
model = large_model()

# Afficher un résumé de l'architecture du modèle (optionnel)
model.summary()

# Entraîner le modèle sur les données d'entraînement,
# avec validation sur les données de test,
# pour 10 époques et des batches de 200 échantillons
history = model.fit(X_train, y_train,
                    validation_data=(X_test, y_test),
                    epochs=10,
                    batch_size=200,
                    verbose=1)  # Afficher la progression

# Évaluer le modèle sur les données de test
scores = model.evaluate(X_test, y_test, verbose=0)

# Afficher le taux d'erreur du modèle (en %)
print(f"Taux d'erreur du modèle : {100 - scores[1]*100:.2f}%")

# Sauvegarder le modèle entraîné au format HDF5
model.save("save_model/large_model_cnn-V0.h5")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entraînement', color='blue', marker='o')
plt.plot(history.history['val_accuracy'], label='Validation', color='orange', marker='v')
plt.title('Précision du modèle (Accuracy)')
plt.xlabel('Époques')
plt.ylabel('Précision')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entraînement', color='blue', marker='o')
plt.plot(history.history['val_loss'], label='Validation', color='orange', marker='v')
plt.title('Perte du modèle (Loss)')
plt.xlabel('Époques')
plt.ylabel('Erreur')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

y_pred = model.predict(X_test)

y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))

plt.title('Matrice de Confusion - MNIST')
plt.ylabel('Vraie classe (Label réel)')
plt.xlabel('Classe prédite par le CNN')
plt.show()

In [ ]:
# =============================================
# Optimisation d'un CNN pour MNIST avec Differential Evolution - VF
# =============================================

# --- 1. Import des bibliothèques nécessaires ---
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential  # Pour créer un modèle séquentiel de couches
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense  # Couches du CNN
from tensorflow.keras.optimizers import Adam  # Optimiseur Adam
from tensorflow.keras.datasets import mnist  # Jeu de données MNIST
from tensorflow.keras.utils import to_categorical  # Pour le one-hot encoding
from scipy.optimize import differential_evolution  # Algorithme d'optimisation, ici c'est Differential Evolution
from sklearn.model_selection import train_test_split  # Pour diviser les données
import warnings
warnings.filterwarnings('ignore')  # Désactive les warnings pour un affichage plus propre

# --- 2. Configuration initiale ---
# Définir explicitement le format des données d'image comme 'channels_last'
# Cela signifie que les tenseurs auront la forme (batch, height, width, channels)
# C'est le format par défaut dans TensorFlow et le plus couramment utilisé
tf.keras.backend.set_image_data_format('channels_last')

# --- 3. Préparation des données MNIST ---
# Charger les données MNIST (images 28x28 de chiffres manuscrits)
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Redimensionner les images pour le format channels_last et normaliser les pixels [0-1]
# -1 dans reshape signifie "calculer automatiquement cette dimension"
# On divise par 255 pour normaliser les pixels entre 0 et 1
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# Convertir les labels en one-hot encoding (ex: 5 → [0,0,0,0,0,1,0,0,0,0])
# Cela permet d'utiliser categorical_crossentropy comme fonction de perte
y_train_cat = to_categorical(y_train, 10)  # 10 classes (chiffres 0-9)
y_test_cat = to_categorical(y_test, 10)

# Diviser les données d'entraînement en ensemble d'entraînement (80%) et validation (20%)
# random_state=42 pour la reproductibilité
X_train, X_val, y_train_cat, y_val_cat = train_test_split(
    X_train, y_train_cat, test_size=0.2, random_state=42
)

# --- 4. Fonction pour créer le modèle CNN ---
def create_model(params):
    """
    Crée un modèle CNN avec les hyperparamètres donnés par Differential Evolution

    Args:
        params: Liste des hyperparamètres à optimiser dans l'ordre:
            [nb_filtres1, taille_noyau1, nb_filtres2, taille_noyau2,
             taux_dropout, neurones_dense1, neurones_dense2, log10_lr]

    Returns:
        model: Modèle Keras compilé
    """
    model = Sequential()  # Initialiser un modèle séquentiel

    # --- Couche 1: Convolution ---
    # Ajouter une couche de convolution 2D avec:
    # - params[0]: nombre de filtres (arrondi à l'entier le plus proche)
    # - params[1]: taille du noyau (carré)
    # - padding='same': conserve les dimensions spatiales (28x28 → 28x28)
    # - activation='relu': fonction d'activation ReLU
    # - input_shape: forme des images d'entrée (28x28 pixels, 1 canal)
    model.add(Conv2D(
        filters=int(round(params[0])),
        kernel_size=(int(round(params[1])), int(round(params[1]))),
        activation='relu',
        input_shape=(28, 28, 1),
        padding='same'  # Important pour conserver les dimensions
    ))
    # Après cette couche: (None, 28, 28, nb_filtres1)

    # Couche de max-pooling pour réduire la taille spatiale
    # pool_size=(2,2) divise la hauteur et largeur par 2
    # Après cette couche: (None, 14, 14, nb_filtres1)
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # --- Couche 2: Convolution ---
    # Deuxième couche de convolution similaire à la première
    model.add(Conv2D(
        filters=int(round(params[2])),
        kernel_size=(int(round(params[3])), int(round(params[3]))),
        activation='relu',
        padding='same'  # Conserve les dimensions (14x14 → 14x14)
    ))
    # Après cette couche: (None, 14, 14, nb_filtres2)

    # Deuxième couche de max-pooling
    # Après cette couche: (None, 7, 7, nb_filtres2)
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # --- Couche 3: Dropout ---
    # Couche de dropout pour la régularisation
    # params[4]: taux de dropout (proportion de neurones désactivés aléatoirement)
    # max/min pour garantir que le taux reste dans [0.1, 0.5]
    model.add(Dropout(max(0.1, min(0.5, params[4]))))

    # --- Couche 4: Flatten ---
    # Aplanir les features pour les couches denses
    # Transforme (None, 7, 7, nb_filtres2) en (None, 7*7*nb_filtres2)
    model.add(Flatten())

    # --- Couche 5: Dense (cachée) ---
    # Couche dense avec params[5] neurones et activation ReLU
    model.add(Dense(int(round(params[5])), activation='relu'))

    # --- Couche 6: Dense (cachée) ---
    # Deuxième couche dense avec params[6] neurones
    model.add(Dense(int(round(params[6])), activation='relu'))

    # --- Couche 7: Sortie ---
    # Couche de sortie avec 10 neurones (un par classe) et softmax
    model.add(Dense(10, activation='softmax'))

    # --- Configuration de l'optimiseur ---
    # Taux d'apprentissage = 10^params[7] (params[7] est en log10)
    # np.clip pour garantir que le taux reste entre 1e-5 et 1e-3
    learning_rate = 10 ** np.clip(params[7], -5, -3)
    optimizer = Adam(learning_rate=learning_rate)

    # --- Compilation du modèle ---
    # loss: categorical_crossentropy pour la classification multi-classe
    # metrics: accuracy pour évaluer les performances
    model.compile(
        loss='categorical_crossentropy',
        optimizer=optimizer,
        metrics=['accuracy']
    )

    return model

# --- 5. Fonction d'évaluation pour Differential Evolution ---
def evaluate_model(params):
    """
    Évalue un modèle avec les hyperparamètres donnés.
    Differential Evolution minimise cette fonction.

    Args:
        params: Hyperparamètres à tester

    Returns:
        1 - accuracy: Pour minimiser (DE minimise la fonction objectif)
    """
    try:
        model = create_model(params)

        # Désactive l'affichage du summary pour éviter le spam
        model.summary(print_fn=lambda x: None)

        # Entraînement rapide (2 époques) pour l'optimisation
        model.fit(
            X_train, y_train_cat,
            epochs=2,
            batch_size=128,
            verbose=0  # Pas d'affichage pendant l'entraînement
        )

        # Évaluation sur l'ensemble de validation
        _, accuracy = model.evaluate(X_val, y_val_cat, verbose=0)
        return 1 - accuracy  # DE minimise, donc on retourne 1-accuracy

    except Exception as e:
        # En cas d'erreur, afficher un message et retourner une mauvaise valeur
        print(f"\nErreur avec params {params}: {str(e)[:200]}...")
        return 1  # Retourne une valeur élevée (mauvaise) pour que DE évite ces params

# --- 6. Définition des bornes pour les hyperparamètres ---
# Chaque tuple définit [min, max] pour un hyperparamètre:
bounds = [
    (10, 30),   # Nombre de filtres couche 1 (entre 10 et 30)
    (2, 3),     # Taille noyau couche 1 : taille des filtres (2x2 ou 3x3)
    (10, 20),   # Nombre de filtres couche 2 (entre 10 et 20)
    (2, 3),     # Taille noyau couche 2 (2x2 ou 3x3)
    (0.1, 0.3), # Taux de dropout (entre 10% et 30%)
    (32, 64),   # Neurones couche dense 1 (entre 32 et 64)
    (16, 32),   # Neurones couche dense 2 (entre 16 et 32)
    (-5, -3)    # log10(taux d'apprentissage) → entre 1e-5 et 1e-3
]

# --- 7. Exécution de Differential Evolution ---
print("Début de l'optimisation par Differential Evolution...")
result = differential_evolution(
    evaluate_model,  # Fonction à optimiser
    bounds,          # Bornes pour chaque paramètre
    strategy='best1bin',  # Stratégie de mutation
    maxiter=3,       # Nombre d'itérations (générations)
    popsize=3,       # Taille de la population (nombre de solutions par génération)
    tol=0.01,        # Tolérance pour la convergence
    mutation=(0.5, 1),  # Facteur de mutation
    recombination=0.7,  # Probabilité de recombination
    seed=42,         # Graine aléatoire pour la reproductibilité
    disp=True        # Afficher la progression
)

# --- 8. Affichage des meilleurs hyperparamètres trouvés ---
best_params = result.x  # Récupérer les meilleurs paramètres
print("\n=== Meilleurs hyperparamètres trouvés ===")
print(f"Nombre de filtres (1ère couche): {int(round(best_params[0]))}")
print(f"Taille noyau (1ère couche): {int(round(best_params[1]))}x{int(round(best_params[1]))}")
print(f"Nombre de filtres (2ème couche): {int(round(best_params[2]))}")
print(f"Taille noyau (2ème couche): {int(round(best_params[3]))}x{int(round(best_params[3]))}")
print(f"Taux de dropout: {best_params[4]:.3f}")
print(f"Neurones (1ère couche dense): {int(round(best_params[5]))}")
print(f"Neurones (2ème couche dense): {int(round(best_params[6]))}")
print(f"Taux d'apprentissage: {10**best_params[7]:.6f}")

# --- 9. Entraînement du modèle final ---
print("\nEntraînement du modèle final avec les meilleurs paramètres...")
final_model = create_model(best_params)
history = final_model.fit(
    X_train, y_train_cat,
    epochs=5,  # Plus d'époques pour le modèle final
    batch_size=128,
    validation_data=(X_val, y_val_cat),
    verbose=1  # Afficher la progression
)

# --- 10. Évaluation finale sur l'ensemble de test ---
test_loss, test_acc = final_model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\n=== Résultats finaux ===")
print(f"Précision sur l'ensemble de test: {test_acc*100:.2f}%")

# --- 11. Sauvegarde du modèle optimisé ---
final_model.save("mnist_model_final.h5")
print("Modèle sauvegardé sous 'mnist_model_final.h5'")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
# Création d'une figure à deux volets
plt.figure(figsize=(12, 4))

# Graphe de la Précision (Accuracy)
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Évolution de la Précision')
plt.xlabel('Époques')
plt.ylabel('Précision')
plt.legend()

# Graphe de la Perte (Loss)
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Évolution de la Perte')
plt.xlabel('Époques')
plt.ylabel('Perte')
plt.legend()

plt.show()



y_pred = final_model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Matrice de Confusion - MNIST')
plt.ylabel('Vraie étiquette')
plt.xlabel('Étiquette prédite')
plt.show()

Méthode GWO avec suil

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns  # <--- AJOUTÉ
from sklearn.metrics import confusion_matrix # <--- AJOUTÉ
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# ==========================================
# 1. PRÉPARATION DES DONNÉES ET CONFIG
# ==========================================

# Configuration
K.set_image_data_format('channels_first')
seed = 42
np.random.seed(seed)
random.seed(seed)

print("Chargement des données MNIST...")
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalisation et Reshape
X_train = X_train.reshape(X_train.shape[0], 1, 28, 28).astype('float32') / 255
X_test = X_test.reshape(X_test.shape[0], 1, 28, 28).astype('float32') / 255

# One-hot encoding
y_train = to_categorical(y_train, num_classes=10)
y_test = to_categorical(y_test, num_classes=10)
num_classes = 10

# ==========================================
# 2. MODÈLE ET FITNESS
# ==========================================

def build_dynamic_model(filters_1, filters_2, dense_1, dropout_rate):
    """Construit un modèle Keras selon les paramètres fournis par le Loup"""
    # Conversion en entiers car les hyperparamètres structurels ne peuvent pas être flottants
    f1 = int(filters_1)
    f2 = int(filters_2)
    d1 = int(dense_1)

    model = Sequential()
    # Couche 1
    model.add(Conv2D(f1, (5, 5), input_shape=(1, 28, 28), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Couche 2
    model.add(Conv2D(f2, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Couches Denses
    model.add(Dropout(dropout_rate))
    model.add(Flatten())
    model.add(Dense(d1, activation='relu'))
    model.add(Dense(50, activation='relu')) # Fixe pour simplifier
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

def fitness_function(position):
    """
    Évalue un loup.
    Position = [Filtres1, Filtres2, Dense1, Dropout, BatchSize]
    Retourne : L'erreur (1 - Accuracy) sur le set de validation.
    """
    # Décodage et bornage des paramètres
    f1 = np.clip(position[0], 16, 64)
    f2 = np.clip(position[1], 8, 32)
    d1 = np.clip(position[2], 64, 256)
    drop = np.clip(position[3], 0.1, 0.5)
    batch = int(np.clip(position[4], 32, 256))

    print(f"   Testing: F1={int(f1)}, F2={int(f2)}, D1={int(d1)}, Drop={drop:.2f}, Batch={batch}...", end="")

    try:
        model = build_dynamic_model(f1, f2, d1, drop)

        # Entraînement COURT pour l'évaluation (2 époques suffisent pour voir la tendance)
        # Cela permet d'aller vite. On fera un entraînement long à la toute fin.
        history = model.fit(X_train, y_train,
                            validation_data=(X_test, y_test),
                            epochs=2,
                            batch_size=batch,
                            verbose=0)

        val_acc = history.history['val_accuracy'][-1]
        error = 1 - val_acc
        print(f" Done. -> Val Acc: {val_acc:.4f}")
        return error

    except Exception as e:
        print(f" Error: {e}")
        return 1.0 # Pénalité maximale

# ==========================================
# 3. ALGORITHME GWO AVEC ARRÊT PRÉCOCE
# ==========================================

def gwo_optimization(search_agents_no, max_iter_safety, lb, ub, dim, patience=3, min_delta=0.001):

    # Initialisation des loups
    positions = np.random.uniform(0, 1, (search_agents_no, dim)) * (ub - lb) + lb

    # Initialisation des Leaders
    Alpha_pos = np.zeros(dim); Alpha_score = float("inf")
    Beta_pos = np.zeros(dim);  Beta_score = float("inf")
    Delta_pos = np.zeros(dim); Delta_score = float("inf")

    # Variables pour l'arrêt précoce
    last_best_score = float("inf")
    patience_counter = 0
    history_scores = []

    print("\n>>> DÉMARRAGE DE L'OPTIMISATION GWO <<<")
    print(f"Agents: {search_agents_no} | Patience: {patience} itérations sans amélioration > {min_delta}")

    for l in range(max_iter_safety):
        print(f"\n--- Itération GWO {l+1} ---")

        # A. Évaluation de tous les loups
        for i in range(search_agents_no):
            # Clip pour rester dans les bornes
            positions[i] = np.clip(positions[i], lb, ub)

            # Calcul du fitness
            fitness = fitness_function(positions[i])

            # Mise à jour Alpha, Beta, Delta
            if fitness < Alpha_score:
                Alpha_score = fitness
                Alpha_pos = positions[i].copy()
            elif fitness < Beta_score:
                Beta_score = fitness
                Beta_pos = positions[i].copy()
            elif fitness < Delta_score:
                Delta_score = fitness
                Delta_pos = positions[i].copy()

        print(f"Meilleure erreur actuelle (Alpha) : {Alpha_score:.4f} (Accuracy: {(1-Alpha_score)*100:.2f}%)")
        history_scores.append(Alpha_score)

        # B. Vérification de la CONVERGENCE (Arrêt précoce)
        improvement = last_best_score - Alpha_score

        if improvement > min_delta:
            print(f" -> Amélioration significative (-{improvement:.4f}). Reset patience.")
            last_best_score = Alpha_score
            patience_counter = 0
        else:
            patience_counter += 1
            print(f" -> Pas d'amélioration suffisante. Patience: {patience_counter}/{patience}")

            if patience_counter >= patience:
                print("\n>>> CONVERGENCE ATTEINTE : Arrêt de l'optimisation. <<<")
                break

        # C. Mise à jour des positions (Mathématiques GWO)
        a = 2 - l * ((2) / max_iter_safety) # Décroissance linéaire de a

        for i in range(search_agents_no):
            for j in range(dim):
                # Formules pour Alpha
                r1, r2 = random.random(), random.random()
                A1 = 2 * a * r1 - a
                C1 = 2 * r2
                D_alpha = abs(C1 * Alpha_pos[j] - positions[i, j])
                X1 = Alpha_pos[j] - A1 * D_alpha

                # Formules pour Beta
                r1, r2 = random.random(), random.random()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2
                D_beta = abs(C2 * Beta_pos[j] - positions[i, j])
                X2 = Beta_pos[j] - A2 * D_beta

                # Formules pour Delta
                r1, r2 = random.random(), random.random()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2
                D_delta = abs(C3 * Delta_pos[j] - positions[i, j])
                X3 = Delta_pos[j] - A3 * D_delta

                # Moyenne
                positions[i, j] = (X1 + X2 + X3) / 3

    return Alpha_pos, Alpha_score, history_scores

# ==========================================
# 4. EXÉCUTION PRINCIPALE
# ==========================================

# Définition de l'espace de recherche
# [Filtres1, Filtres2, Dense1, Dropout, BatchSize]
lb = np.array([16,  8,  64, 0.1,  32]) # Borne inférieure
ub = np.array([64, 32, 256, 0.5, 256]) # Borne supérieure
dim = 5

# Paramètres de l'algo
# Note : Mettez search_agents_no=5 ou plus pour de bons résultats (mais c'est plus lent)
search_agents = 5
max_safety_iter = 20 # Sécurité max
patience_val = 7    # Arrêt si pas mieux après 3 tours

# Lancement GWO
best_params, best_score, score_history = gwo_optimization(search_agents, max_safety_iter, lb, ub, dim, patience=patience_val)

# Affichage des résultats
print("\n" + "="*40)
print(f"MEILLEURE CONFIGURATION TROUVÉE")
print("="*40)
print(f"Filtres Couche 1 : {int(best_params[0])}")
print(f"Filtres Couche 2 : {int(best_params[1])}")
print(f"Neurones Dense 1 : {int(best_params[2])}")
print(f"Taux Dropout     : {best_params[3]:.4f}")
print(f"Batch Size       : {int(best_params[4])}")
print(f"Précision estimée: {(1-best_score)*100:.2f}%")
print("="*40)

# ==========================================
# 5. ENTRAÎNEMENT FINAL (VALIDATION)
# ==========================================
print("\nEntraînement du modèle FINAL avec ces paramètres (plus d'époques)...")

final_model = build_dynamic_model(best_params[0], best_params[1], best_params[2], best_params[3])

# Callback pour sauvegarder le meilleur modèle
from tensorflow.keras.callbacks import ModelCheckpoint
checkpoint = ModelCheckpoint("best_gwo_cnn.keras", monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')

final_history = final_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=25, # On entraîne plus longtemps pour avoir la vraie performance
    batch_size=int(best_params[4]),
    callbacks=[checkpoint],
    verbose=1
)

# ==========================================
# 6. VISUALISATION DES RÉSULTATS
# ==========================================
print("Génération des graphiques...")

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(score_history, 'g-o')
plt.title('Convergence du GWO avec seuil')
plt.xlabel('Itérations')
plt.ylabel('Erreur (Loss)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(final_history.history['accuracy'], label='Train Accuracy')
plt.plot(final_history.history['val_accuracy'], label='Val Accuracy')
plt.title('Performance du Modèle Final avec seuil')
plt.xlabel('Époques')
plt.ylabel('Précision')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


plt.subplot(1, 2, 2)
plt.plot(final_history.history['loss'], label='Train Loss')
plt.plot(final_history.history['val_loss'], label='Val Loss')
plt.title(' Évolution de la Perte (Loss)')
plt.xlabel('Époques')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


plt.figure(figsize=(20,12))

plt.subplot(2, 3, 5)
# Prédictions sur le jeu de test
y_pred_prob = final_model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1) # Convertir probas en classes (0-9)
if y_test.ndim > 1: y_test_classes = np.argmax(y_test, axis=1)
else: y_test_classes = y_test

cm = confusion_matrix(y_test_classes, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('5. Matrice de Confusion', fontsize=14)
plt.xlabel('Prédit')
plt.ylabel('Réel')

plt.tight_layout()
plt.show()



# Score final sur le test set
loss, acc = final_model.evaluate(X_test, y_test, verbose=0)
print(f"\n>>> RÉSULTAT FINAL SUR LE JEU DE TEST : {acc*100:.2f}% <<<")

Méthode GWO avec itérations

In [ ]:
# Le code final optimisé par la méthode Grey Wolf Optimizer ( avec itérations)

# bibliothèques nécessaires
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential  # Pour créer un modèle séquentiel de couches
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense  # Couches du CNN
from tensorflow.keras.optimizers import Adam  # Optimiseur Adam
from tensorflow.keras.datasets import mnist  # Jeu de données MNIST
from tensorflow.keras.utils import to_categorical  # Pour le one-hot encoding
from scipy.optimize import differential_evolution  # Algorithme d'optimisation, ici c'est Differential Evolution
from sklearn.model_selection import train_test_split  # Pour diviser les données
import warnings
import seaborn as sns
from sklearn.metrics import confusion_matrix

warnings.filterwarnings('ignore')  # Désactive les warnings pour un affichage plus propre

# --- 2. Configuration initiale ---
# Définir explicitement le format des données d'image comme 'channels_last'
# Cela signifie que les tenseurs auront la forme (batch, height, width, channels)
# C'est le format par défaut dans TensorFlow et le plus couramment utilisé
tf.keras.backend.set_image_data_format('channels_last')

# --- 3. Préparation des données MNIST ---
# Charger les données MNIST (images 28x28 de chiffres manuscrits)
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Redimensionner les images pour le format channels_last et normaliser les pixels [0-1]
# -1 dans reshape signifie "calculer automatiquement cette dimension"
# On divise par 255 pour normaliser les pixels entre 0 et 1
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# Convertir les labels en one-hot encoding (ex: 5 → [0,0,0,0,0,1,0,0,0,0])
# Cela permet d'utiliser categorical_crossentropy comme fonction de perte
y_train_cat = to_categorical(y_train, 10)  # 10 classes (chiffres 0-9)
y_test_cat = to_categorical(y_test, 10)

# Diviser les données d'entraînement en ensemble d'entraînement (80%) et validation (20%)
# random_state=42 pour la reproductibilité
X_train, X_val, y_train_cat, y_val_cat = train_test_split(
    X_train, y_train_cat, test_size=0.2, random_state=42
)

# --- 4. Fonction pour créer le modèle CNN ---
def create_model(params):
    """
    Crée un modèle CNN avec les hyperparamètres donnés par Differential Evolution

    Args:
        params: Liste des hyperparamètres à optimiser dans l'ordre:
            [nb_filtres1, taille_noyau1, nb_filtres2, taille_noyau2,
             taux_dropout, neurones_dense1, neurones_dense2, log10_lr]

    Returns:
        model: Modèle Keras compilé
    """
    model = Sequential()  # Initialiser un modèle séquentiel

    # --- Couche 1: Convolution ---
    # Ajouter une couche de convolution 2D avec:
    # - params[0]: nombre de filtres (arrondi à l'entier le plus proche)
    # - params[1]: taille du noyau (carré)
    # - padding='same': conserve les dimensions spatiales (28x28 → 28x28)
    # - activation='relu': fonction d'activation ReLU
    # - input_shape: forme des images d'entrée (28x28 pixels, 1 canal)
    model.add(Conv2D(
        filters=int(round(params[0])),
        kernel_size=(int(round(params[1])), int(round(params[1]))),
        activation='relu',
        input_shape=(28, 28, 1),
        padding='same'  # Important pour conserver les dimensions
    ))
    # Après cette couche: (None, 28, 28, nb_filtres1)

    # Couche de max-pooling pour réduire la taille spatiale
    # pool_size=(2,2) divise la hauteur et largeur par 2
    # Après cette couche: (None, 14, 14, nb_filtres1)
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # --- Couche 2: Convolution ---
    # Deuxième couche de convolution similaire à la première
    model.add(Conv2D(
        filters=int(round(params[2])),
        kernel_size=(int(round(params[3])), int(round(params[3]))),
        activation='relu',
        padding='same'  # Conserve les dimensions (14x14 → 14x14)
    ))
    # Après cette couche: (None, 14, 14, nb_filtres2)

    # Deuxième couche de max-pooling
    # Après cette couche: (None, 7, 7, nb_filtres2)
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # --- Couche 3: Dropout ---
    # Couche de dropout pour la régularisation
    # params[4]: taux de dropout (proportion de neurones désactivés aléatoirement)
    # max/min pour garantir que le taux reste dans [0.1, 0.5]
    model.add(Dropout(max(0.1, min(0.5, params[4]))))

    # --- Couche 4: Flatten ---
    # Aplanir les features pour les couches denses
    # Transforme (None, 7, 7, nb_filtres2) en (None, 7*7*nb_filtres2)
    model.add(Flatten())

    # --- Couche 5: Dense (cachée) ---
    # Couche dense avec params[5] neurones et activation ReLU
    model.add(Dense(int(round(params[5])), activation='relu'))

    # --- Couche 6: Dense (cachée) ---
    # Deuxième couche dense avec params[6] neurones
    model.add(Dense(int(round(params[6])), activation='relu'))

    # --- Couche 7: Sortie ---
    # Couche de sortie avec 10 neurones (un par classe) et softmax
    model.add(Dense(10, activation='softmax'))

    # --- Configuration de l'optimiseur ---
    # Taux d'apprentissage = 10^params[7] (params[7] est en log10)
    # np.clip pour garantir que le taux reste entre 1e-5 et 1e-3
    learning_rate = 10 ** np.clip(params[7], -5, -3)
    optimizer = Adam(learning_rate=learning_rate)

    # --- Compilation du modèle ---
    # loss: categorical_crossentropy pour la classification multi-classe
    # metrics: accuracy pour évaluer les performances
    model.compile(
        loss='categorical_crossentropy',
        optimizer=optimizer,
        metrics=['accuracy']
    )

    return model

# --- 5. Fonction d'évaluation pour Differential Evolution ---
def evaluate_model(params):
    """
    Évalue un modèle avec les hyperparamètres donnés.
    Differential Evolution minimise cette fonction.

    Args:
        params: Hyperparamètres à tester

    Returns:
        1 - accuracy: Pour minimiser (DE minimise la fonction objectif)
    """
    try:
        model = create_model(params)

        # Désactive l'affichage du summary pour éviter le spam
        model.summary(print_fn=lambda x: None)

        # Entraînement rapide (2 époques) pour l'optimisation
        model.fit(
            X_train, y_train_cat,
            epochs=2,
            batch_size=128,
            verbose=0  # Pas d'affichage pendant l'entraînement
        )

        # Évaluation sur l'ensemble de validation
        _, accuracy = model.evaluate(X_val, y_val_cat, verbose=0)
        return 1 - accuracy  # DE minimise, donc on retourne 1-accuracy

    except Exception as e:
        # En cas d'erreur, afficher un message et retourner une mauvaise valeur
        print(f"\nErreur avec params {params}: {str(e)[:200]}...")
        return 1  # Retourne une valeur élevée (mauvaise) pour que DE évite ces params

# --- 6. Définition des bornes pour les hyperparamètres ---
# Chaque tuple définit [min, max] pour un hyperparamètre:
bounds = [
    (10, 30),   # Nombre de filtres couche 1 (entre 10 et 30)
    (2, 3),     # Taille noyau couche 1 : taille des filtres (2x2 ou 3x3)
    (10, 20),   # Nombre de filtres couche 2 (entre 10 et 20)
    (2, 3),     # Taille noyau couche 2 (2x2 ou 3x3)
    (0.1, 0.3), # Taux de dropout (entre 10% et 30%)
    (32, 64),   # Neurones couche dense 1 (entre 32 et 64)
    (16, 32),   # Neurones couche dense 2 (entre 16 et 32)
    (-5, -3)    # log10(taux d'apprentissage) → entre 1e-5 et 1e-3
]


# implémentation du GREY WOLF OPTIMIZER (GWO)
# 1. GWO


def grey_wolf_optimizer_tracked(obj_func, bounds, pop_size=5, max_iter=5):

    dim = len(bounds)
    lb = np.array([b[0] for b in bounds])
    ub = np.array([b[1] for b in bounds])

    # Initialisation
    positions = np.random.uniform(low=lb, high=ub, size=(pop_size, dim))
    alpha_pos = np.zeros(dim); alpha_score = float("inf")
    beta_pos = np.zeros(dim); beta_score = float("inf")
    delta_pos = np.zeros(dim); delta_score = float("inf")

    # HISTORIQUES POUR GRAPHIQUES
    history_score = []       # Pour la courbe de convergence
    history_positions = []   # Pour le scatter plot (position de tous les loups)
    history_best_pos = []    # On sauvegarde la position du chef
    print(f"Début du GWO avec {pop_size} loups sur {max_iter} itérations...")

    for t in range(max_iter):
        # Sauvegarde des positions actuelles (pour le scatter plot)
        history_positions.append(positions.copy())

        # A. Évaluation
        for i in range(pop_size):
            positions[i] = np.clip(positions[i], lb, ub)
            fitness = obj_func(positions[i])

            print(f"  > Iter {t+1} | Loup {i+1} | Score: {fitness:.4f}")

            # Mise à jour Alpha, Beta, Delta
            if fitness < alpha_score:
                alpha_score = fitness; alpha_pos = positions[i].copy()
            elif fitness < beta_score:
                beta_score = fitness; beta_pos = positions[i].copy()
            elif fitness < delta_score:
                delta_score = fitness; delta_pos = positions[i].copy()

        # Sauvegarde du meilleur score actuel (pour la courbe de convergence)
        history_score.append(alpha_score)
        history_best_pos.append(alpha_pos.copy())
        # B. Mise à jour des positions
        a = 2 - t * (2 / max_iter)
        for i in range(pop_size):
            for j in range(dim):
                r1, r2 = np.random.random(), np.random.random()
                A1 = 2*a*r1 - a; C1 = 2*r2
                D_alpha = abs(C1*alpha_pos[j] - positions[i,j])
                X1 = alpha_pos[j] - A1*D_alpha

                r1, r2 = np.random.random(), np.random.random()
                A2 = 2*a*r1 - a; C2 = 2*r2
                D_beta = abs(C2*beta_pos[j] - positions[i,j])
                X2 = beta_pos[j] - A2*D_beta

                r1, r2 = np.random.random(), np.random.random()
                A3 = 2*a*r1 - a; C3 = 2*r2
                D_delta = abs(C3*delta_pos[j] - positions[i,j])
                X3 = delta_pos[j] - A3*D_delta

                positions[i,j] = (X1 + X2 + X3) / 3

        print(f"--- Fin itération {t+1} --- Meilleure erreur : {alpha_score:.4f}\n")

    return alpha_pos, history_score, history_positions, history_best_pos

# Lancement
best_params, scores_history, positions_history, history_best_pos = grey_wolf_optimizer_tracked(
    evaluate_model,
    bounds,
    pop_size=12,
    max_iter=10
)

class ResultObject:
    pass
result = ResultObject()
result.x = best_params

# 8. Affichage des meilleurs hyperparamètres trouvés
best_params = result.x  # Récupérer les meilleurs paramètres
print("\n=== Meilleurs hyperparamètres trouvés ===")
print(f"Nombre de filtres (1ère couche): {int(round(best_params[0]))}")
print(f"Taille noyau (1ère couche): {int(round(best_params[1]))}x{int(round(best_params[1]))}")
print(f"Nombre de filtres (2ème couche): {int(round(best_params[2]))}")
print(f"Taille noyau (2ème couche): {int(round(best_params[3]))}x{int(round(best_params[3]))}")
print(f"Taux de dropout: {best_params[4]:.3f}")
print(f"Neurones (1ère couche dense): {int(round(best_params[5]))}")
print(f"Neurones (2ème couche dense): {int(round(best_params[6]))}")
print(f"Taux d'apprentissage: {10**best_params[7]:.6f}")

# 9. Entraînement du modèle final
print("\nEntraînement du modèle final avec les meilleurs paramètres...")

# Ajout d'un "Early Stopping" pour arrêter quand c'est cuit (Optionnel mais classe)
from tensorflow.keras.callbacks import EarlyStopping
arret_auto = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

final_model = create_model(best_params)

history = final_model.fit(
    X_train, y_train_cat,
    epochs=30,
    batch_size=128,
    validation_data=(X_val, y_val_cat),
    #callbacks=[arret_auto], # <--- Ajout de l'arrêt auto (supprimez cette ligne si vous ne voulez pas l'utiliser)
    verbose=1
)
# 10. Évaluation finale sur l'ensemble de test
test_loss, test_acc = final_model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\n=== Résultats finaux ===")
print(f"Précision sur l'ensemble de test: {test_acc*100:.2f}%")

# 11. Sauvegarde du modèle optimisé
final_model.save("mnist_model_final.h5")
print("Modèle sauvegardé sous 'mnist_model_final.h5'")

final_model.summary()

In [ ]:
# 2. GÉNÉRATION DES 4 GRAPHIQUES

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

plt.figure(figsize=(20, 12))

# GRAPHIQUE 1 : Courbe de Convergence
plt.subplot(2, 3, 1)
plt.plot(scores_history, 'b-o', linewidth=2, label='Meilleur Loup (Alpha)')
plt.title('1. Convergence du GWO', fontsize=14)
plt.xlabel('Itérations')
plt.ylabel('Erreur (1 - Accuracy)')
plt.grid(True)
plt.legend()

# GRAPHIQUE 2 : Exploration 2D (Scatter Plot)
# On visualise l'évolution sur 2 paramètres:
plt.subplot(2, 3, 2)
pos_ini = positions_history[0]   # Positions initiales
pos_fin = positions_history[-1]  # Positions finales

# Axe X: Param 0 (Nb Filtres), Axe Y: Param 7 (Learning Rate)
plt.scatter(pos_ini[:, 0], pos_ini[:, 7], color='red', alpha=0.7, label='Début (Exploration)')
plt.scatter(pos_fin[:, 0], pos_fin[:, 7], color='blue', alpha=0.7, label='Fin (Exploitation)')
# Afficher le leader final
plt.scatter(best_params[0], best_params[7], color='green', s=100, marker='*', label='Alpha Final')

plt.title('2. Comportement de la Meute', fontsize=14)
plt.xlabel('Nb Filtres (Couche 1)')
plt.ylabel('Log Learning Rate')
plt.legend()
plt.grid(True)

# GRAPHIQUE 3 : Courbes d'Apprentissage (Loss)
# Utilise l'historique de l'entraînement final 'history'
plt.subplot(2, 3, 3)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('3. Apprentissage du Meilleur CNN', fontsize=14)
plt.xlabel('Époques')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)


# GRAPHIQUE 4 : Matrice de Confusion
plt.subplot(2, 3, 5)
# Prédictions sur le jeu de test
y_pred_prob = final_model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1) # Convertir probas en classes (0-9)
if y_test.ndim > 1: y_test_classes = np.argmax(y_test, axis=1)
else: y_test_classes = y_test

cm = confusion_matrix(y_test_classes, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('5. Matrice de Confusion', fontsize=14)
plt.xlabel('Prédit')
plt.ylabel('Réel')

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CRÉATION DE L'ANIMATION (GIF)
# ==========================================
!pip install pillow
import matplotlib.animation as animation
print("Génération de l'animation en cours...")

# 1. Configuration de la figure
fig, ax = plt.subplots(figsize=(10, 6))

# On choisit deux paramètres à visualiser (comme pour le Scatter Plot précédent)
# Index 0 : Nombre de filtres (Couche 1)
# Index 7 : Log Learning Rate
idx_x = 0
idx_y = 7
nom_x = "Nb Filtres (Couche 1)"
nom_y = "Log Learning Rate"

# Définir les limites du graphique (pour que ça ne bouge pas)
# On prend les bornes définies dans 'bounds' + une petite marge
x_min, x_max = bounds[idx_x]
y_min, y_max = bounds[idx_y]
ax.set_xlim(x_min - 1, x_max + 1)
ax.set_ylim(y_min - 0.5, y_max + 0.5)

ax.set_xlabel(nom_x)
ax.set_ylabel(nom_y)
ax.grid(True, linestyle='--', alpha=0.6)

# 2. Création des objets graphiques vides
# Les loups (points bleus)
scat = ax.scatter([], [], c='blue', alpha=0.7, s=50, label='Loups (Omega)')
# Le leader (étoile rouge)
leader = ax.scatter([], [], c='red', marker='*', s=200, label='Alpha (Leader)')
# Le titre qui change
title = ax.set_title("")
ax.legend(loc='upper right')

# 3. Fonction de mise à jour (appelée à chaque frame)
def update(frame):
    # Récupérer les positions de tous les loups à l'itération 'frame'
    pos_actuels = positions_history[frame]

    # Mettre à jour les coordonnées X et Y des points bleus
    # On prend la colonne idx_x et idx_y
    data = np.column_stack((pos_actuels[:, idx_x], pos_actuels[:, idx_y]))
    scat.set_offsets(data)

    # Trouver le meilleur loup de cette itération (celui avec le score min)
    # Note: Dans votre code GWO modifié, history_score[frame] donne le meilleur score,
    # mais pour simplifier l'animation, on va dire que le leader est le premier de la liste triée
    # ou simplement afficher la position du meilleur enregistré

    # Pour l'animation, on va tricher légèrement et afficher le loup le plus proche de la solution finale
    # ou recalculer le meilleur de l'itération courante.
    # Ici, on affiche juste les loups, c'est le mouvement de groupe qui compte.
    # 2. Mise à jour de l'étoile rouge (C'est ici que ça manquait !)
    pos_chef = history_best_pos[frame]
    data_chef = np.column_stack((pos_chef[idx_x], pos_chef[idx_y]))
    leader.set_offsets(data_chef)
    title.set_text(f"Itération {frame + 1} / {len(positions_history)}")
    return scat, leader, title

# 4. Lancement de l'animation
# frames = nombre d'étapes enregistrées dans positions_history
anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(positions_history),
    interval=500, # Vitesse : 500ms entre chaque image (0.5 seconde)
    blit=False
)

# 5. Sauvegarde en GIF
nom_fichier = "evolution_gwoo.gif"
try:
    anim.save(nom_fichier, writer='pillow', fps=2)
    print(f"✅ Animation sauvegardée sous : {nom_fichier}")
    print("Vous pouvez télécharger ce fichier et le mettre dans votre PPT.")
except Exception as e:
    print(f"Erreur lors de la sauvegarde : {e}")
    print("Assurez-vous d'avoir 'pillow' installé (pip install pillow)")

plt.show()